# Appendix A1: Chunking strategy study

Not one of "the 10 patterns" -- no §8 template (same exemption as `00_baseline_no_rag.ipynb`/
`00b_long_context_baseline.ipynb`). Holds retrieval constant (hybrid + cross-encoder rerank,
`recipes/hybrid_rerank.py`) and varies only how the corpus is chunked, reporting a paper-level
hit@10 (`evals.metrics.paper_hit_at_k`) per variant.

**Requires the `corpus-build` extra** (`uv sync --extra corpus-build`) for `tiktoken`, unlike the 10
pattern notebooks which need only the base `uv sync` -- this is the one appendix that needs a real
tokenizer to build token-sized chunks.

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`.** What's genuinely real
regardless of mock/real: every chunking variant itself is built from real, full paper text
(`corpus/corpus_fulltext.jsonl`, fetched once via the one-off `corpus/build_fulltext.py` -- see
`tasks/todo.md`), using real chunking logic (`corpus/chunking_strategies.py`), with **zero live
network calls at notebook-execution time**. What's PENDING: the actual retrieval-quality numbers
below need real `text-embedding-3-small` embeddings (and, for the semantic-chunking variant, real
embeddings to find its split points too) -- under mock, every row's `paper_hit@10` reflects
`MockEmbedder`'s non-semantic hash vectors, not real retrieval quality. This is the SAME PENDING
treatment already applied to 8 of the 10 main patterns; nothing about this notebook is more or less
"real" than those.

Note on ground truth: `qa_set.jsonl`'s `relevant_chunk_ids` reference the ORIGINAL corpus's
chunk_ids, which don't exist in any re-chunked variant. `paper_hit_at_k` compares at the PAPER
level instead (`paper_id` is stable across every chunking strategy) -- coarser than the main
leaderboard's `hit@10`, and stated here explicitly rather than silently swapped in.


## Reproducibility header

In [1]:
import platform
import sys
import subprocess
import openai
import numpy

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: e9d172f8a88da3577c09ecf807b8cb38e79db941


## Setup

In [2]:
import os
os.environ.setdefault("RAG_RECIPES_LLM", "mock")

import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from evals.run import load_corpus_by_id, load_qa_set
from evals.metrics import paper_hit_at_k, bootstrap_ci
from recipes.embeddings import get_embedder
from recipes.hybrid_rerank import build_retriever
from corpus.chunking_strategies import (
    chunk_fixed, chunk_semantic, chunk_document_aware, chunk_late,
)

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
embedder = get_embedder()

full_text_by_paper = {}
with open("../corpus/corpus_fulltext.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        full_text_by_paper[rec["paper_id"]] = rec["full_text"]

def _get_tokenizer():
    import tiktoken
    try:
        return tiktoken.encoding_for_model("gpt-4.1-mini")
    except KeyError:
        return tiktoken.get_encoding("cl100k_base")

tokenizer = _get_tokenizer()

# Map each question to its relevant PAPER ids (not chunk ids) -- see the
# top-of-notebook note on why paper_hit_at_k exists.
relevant_paper_ids_by_qid = {
    r["qid"]: {corpus_by_id[cid]["paper_id"] for cid in r["relevant_chunk_ids"] if cid in corpus_by_id}
    for r in qa_set
}

K = 10


In [3]:
def print_results_table(rows):
    """rows: list of (label, ConfidenceInterval, is_real: bool)."""
    print(f"{'variant':<28} {'paper_hit@10':<24} {'status'}")
    for label, ci, is_real in rows:
        ci_str = f"{ci.mean:.3f}  [95% CI {ci.lower:.3f}, {ci.upper:.3f}]"
        status = "REAL" if is_real else "PENDING (mock)"
        print(f"{label:<28} {ci_str:<24} {status}")


## Build each chunking variant's corpus, then score paper_hit@10

In [4]:
def score_variant(variant_corpus_by_id, label):
    retrieve = build_retriever(variant_corpus_by_id, embedder=embedder)
    scores = []
    for r in qa_set:
        relevant_papers = relevant_paper_ids_by_qid[r["qid"]]
        if not relevant_papers:
            continue
        retrieved_ids = retrieve(r["question"], K)
        scores.append(paper_hit_at_k(retrieved_ids, relevant_papers, variant_corpus_by_id, K))
    return bootstrap_ci(scores)

def build_variant_corpus(chunks_per_paper: dict[str, list[dict]]) -> dict[str, dict]:
    """chunks_per_paper: paper_id -> list of {"text", "embed_text"} dicts.
    Returns a corpus_by_id-shaped dict; "text" is what gets embedded AND
    cited (embed_text is only used internally by chunk_late's caller, see
    below), chunk_id is synthesized as f"{paper_id}#{i}".
    """
    out = {}
    for paper_id, chunks in chunks_per_paper.items():
        for i, chunk in enumerate(chunks):
            cid = f"{paper_id}#{i}"
            out[cid] = {"chunk_id": cid, "paper_id": paper_id, "text": chunk["embed_text"]}
    return out

results = []


### fixed-256 / fixed-512 / fixed-1024

In [5]:
for chunk_tokens in (256, 512, 1024):
    chunks_per_paper = {
        paper_id: chunk_fixed(text, chunk_tokens=chunk_tokens, tokenizer=tokenizer)
        for paper_id, text in full_text_by_paper.items()
    }
    variant_corpus = build_variant_corpus(chunks_per_paper)
    ci = score_variant(variant_corpus, f"fixed-{chunk_tokens}")
    results.append((f"fixed-{chunk_tokens}", ci, False))


### semantic (embeddings-based)

In [6]:
chunks_per_paper = {
    paper_id: chunk_semantic(text, embedder=embedder, embedding_model="text-embedding-3-small", tokenizer=tokenizer)
    for paper_id, text in full_text_by_paper.items()
}
variant_corpus = build_variant_corpus(chunks_per_paper)
ci = score_variant(variant_corpus, "semantic")
results.append(("semantic", ci, False))


### document-aware (section-boundary)

In [7]:
chunks_by_paper_original = {}
for chunk in corpus_by_id.values():
    chunks_by_paper_original.setdefault(chunk["paper_id"], []).append(chunk)

chunks_per_paper = {
    paper_id: chunk_document_aware(chunks) for paper_id, chunks in chunks_by_paper_original.items()
}
variant_corpus = build_variant_corpus(chunks_per_paper)
ci = score_variant(variant_corpus, "document-aware")
results.append(("document-aware", ci, False))


### late chunking (context-window approximation -- see top-of-notebook disclaimer)

In [8]:
chunks_per_paper = {
    paper_id: chunk_late(text, chunk_tokens=512, tokenizer=tokenizer)
    for paper_id, text in full_text_by_paper.items()
}
variant_corpus = build_variant_corpus(chunks_per_paper)
ci = score_variant(variant_corpus, "late (approx.)")
results.append(("late (approx.)", ci, False))


## Results

In [9]:
print_results_table(results)

variant                      paper_hit@10             status
fixed-256                    1.000  [95% CI 1.000, 1.000] PENDING (mock)
fixed-512                    0.833  [95% CI 0.667, 1.000] PENDING (mock)
fixed-1024                   1.000  [95% CI 1.000, 1.000] PENDING (mock)
semantic                     0.889  [95% CI 0.722, 1.000] PENDING (mock)
document-aware               0.889  [95% CI 0.722, 1.000] PENDING (mock)
late (approx.)               0.944  [95% CI 0.833, 1.000] PENDING (mock)


## Where this study is incomplete

**PENDING: real retrieval-quality findings.** Every row above is computed under
`RAG_RECIPES_LLM=mock` -- `MockEmbedder`'s hash-based vectors have no real semantic content, so
`paper_hit@10` numbers above only prove the code path runs end to end for all 6 chunking
strategies, not which strategy actually retrieves better. A real run needs `OPENAI_API_KEY`; see
`tasks/todo.md` for the consolidated real-key-run backlog covering this alongside 8 other
PENDING notebooks.
